# SmartRetail – AI Demand Forecasting & Inventory Analytics
### Data Science Module | Python | Pandas | scikit-learn
> Analyzing retail sales data to predict demand, detect slow-moving products, and surface inventory insights.

## Section 1 – Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import urllib.request
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/charts', exist_ok=True)

url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/retail_sales.csv'
urllib.request.urlretrieve(url, 'data/Train.csv')

df = pd.read_csv('data/Train.csv')
print('Shape:', df.shape)
print(df.dtypes)
df.head()

## Section 2 – Data Cleaning

In [ ]:
print('Missing values:\n', df.isnull().sum())

for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('\nAfter cleaning:')
print(df.isnull().sum())
print('Shape:', df.shape)

## Section 3 – Feature Engineering

In [ ]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df['month'] = df['Date'].dt.month
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['quarter'] = df['Date'].dt.quarter

sales_col = [c for c in df.columns if 'sale' in c.lower() or 'Sales' in c][0]
print('Sales column detected:', sales_col)

df['rolling_7day_avg'] = df[sales_col].rolling(7, min_periods=1).mean()
df['rolling_30day_avg'] = df[sales_col].rolling(30, min_periods=1).mean()

df.head()

## Section 4 – EDA & Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SmartRetail – Sales Analytics Dashboard', fontsize=16, fontweight='bold')

axes[0,0].hist(df[sales_col], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Sales Distribution')
axes[0,0].set_xlabel('Sales')

if 'month' in df.columns:
    monthly = df.groupby('month')[sales_col].mean()
    axes[0,1].bar(monthly.index, monthly.values, color='teal')
    axes[0,1].set_title('Average Sales by Month')
    axes[0,1].set_xlabel('Month')
else:
    df[sales_col].plot(ax=axes[0,1], color='teal')
    axes[0,1].set_title('Sales Over Time')

axes[0,2].plot(df[sales_col].values[:200], label='Actual', alpha=0.6)
axes[0,2].plot(df['rolling_7day_avg'].values[:200], label='7-day avg', color='red')
axes[0,2].set_title('Sales Trend (Rolling Average)')
axes[0,2].legend()

cat_col = [c for c in df.columns if 'cat' in c.lower() or 'type' in c.lower() or 'item' in c.lower()]
if cat_col:
    top_cat = df.groupby(cat_col[0])[sales_col].mean().sort_values(ascending=False).head(8)
    top_cat.plot(kind='bar', ax=axes[1,0], color='coral')
    axes[1,0].set_title(f'Avg Sales by {cat_col[0]}')
    axes[1,0].tick_params(axis='x', rotation=45)
else:
    df[sales_col].rolling(30).mean().plot(ax=axes[1,0], color='coral')
    axes[1,0].set_title('30-day Rolling Trend')

if 'is_weekend' in df.columns:
    df.groupby('is_weekend')[sales_col].mean().plot(kind='bar', ax=axes[1,1], color=['steelblue','orange'])
    axes[1,1].set_title('Weekday vs Weekend Sales')
    axes[1,1].set_xticklabels(['Weekday','Weekend'], rotation=0)
else:
    df[sales_col].plot(kind='box', ax=axes[1,1])
    axes[1,1].set_title('Sales Box Plot')

df['is_slow_moving'] = (df[sales_col] < df[sales_col].quantile(0.25)).astype(int)
df['is_slow_moving'].value_counts().plot(kind='pie', ax=axes[1,2],
    labels=['Normal','Slow-Moving'], colors=['green','red'], autopct='%1.1f%%')
axes[1,2].set_title('Slow-Moving Product Share')
axes[1,2].set_ylabel('')

plt.tight_layout()
plt.savefig('outputs/charts/sales_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved to outputs/charts/')

## Section 5 – ML Model (Random Forest Demand Forecasting)

In [ ]:
df_model = df.copy()
le = LabelEncoder()

for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

if 'Date' in df_model.columns:
    df_model.drop('Date', axis=1, inplace=True)

X = df_model.drop(sales_col, axis=1)
y = df_model[sales_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'R²   : {r2:.4f}')

## Section 6 – Feature Importance & Business Insights

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('SmartRetail – Top Demand Drivers (Feature Importance)')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('outputs/charts/feature_importance.png', dpi=150)
plt.show()

slow = df[df['is_slow_moving'] == 1]
print(f'\n📊 Business Insights:')
print(f'1. {len(slow)} products are slow-moving — consider discounts or combo offers.')
print(f'2. Top demand driver: {feat_imp.index[0]} — prioritize stocking decisions around this.')
print(f'3. Model R² = {r2:.2f} — forecasting explains {r2*100:.0f}% of sales variance.')
print(f'4. Average sales: {df[sales_col].mean():.0f} | Peak: {df[sales_col].max():.0f}')

---
## Resume Description
> Built the data science core of SmartRetail, an AI-powered retail inventory system, covering demand forecasting, seasonal trend analysis, waste prediction, and inventory anomaly detection using Python, Pandas, and scikit-learn on retail transaction data.